# Employee Attrition Analytics

## Module 10: Model Persistence and Prediction Pipeline

### Objective

Save the trained machine learning model and use it to make predictions on new employee data.

In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

## Load ML-Ready Dataset

In [2]:
df = pd.read_csv("../data/processed/employee_attrition_ml_ready.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (1000, 52)


,Age,Gender,Experience,Salary,Performance_Score,Job_Satisfaction,Overtime,Work_Hours,Remote_Work,Training_Hours,...,Salary_Band_Medium,Salary_Band_High,Salary_Band_Very High,Experience_Band_Junior,Experience_Band_Mid Level,Experience_Band_Senior,Satisfaction_Category_Medium,Satisfaction_Category_High,Performance_Category_Average,Performance_Category_High
0,40.0,0,3,94113.0,5,6.0,1,46,0,70,...,1,0,0,1,0,0,1,0,0,1
1,39.0,1,2,54224.0,4,9.0,0,36,0,33,...,0,0,0,0,0,0,0,1,0,1
2,49.0,0,25,166749.0,2,6.0,0,48,1,75,...,0,1,0,0,0,1,1,0,0,0
3,54.0,1,26,196590.0,5,4.0,1,51,1,64,...,0,0,1,0,0,1,0,0,0,1
4,41.0,1,18,231764.0,3,8.0,1,40,1,15,...,0,0,1,0,0,1,0,1,1,0


## Prepare Features and Target

In [3]:
X = df.drop(columns=["Attrition", "Attrition_Target"])

y = df["Attrition_Target"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (1000, 50)
Target: (1000,)


## Check Feature Data Types

In [4]:
non_numeric_columns = X.select_dtypes(exclude=np.number).columns

print("Non-numeric columns:")
print(non_numeric_columns.tolist())

Non-numeric columns:
['Promotion_Last_5Yrs']


In [5]:
X["Promotion_Last_5Yrs"] = X["Promotion_Last_5Yrs"].map({"No": 0, "Yes": 1})

print(X["Promotion_Last_5Yrs"].value_counts())

Promotion_Last_5Yrs
0    794
1    206
Name: count, dtype: int64


In [6]:
non_numeric_columns = X.select_dtypes(exclude=np.number).columns

print("Non-numeric columns:")
print(non_numeric_columns.tolist())

Non-numeric columns:
[]


## Train Final Random Forest Model

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
)

final_model.fit(X_train, y_train)

print("Final model trained successfully.")

Final model trained successfully.


## Evaluate Final Model

In [8]:
y_pred = final_model.predict(X_test)

y_probability = final_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred, zero_division=0)

recall = recall_score(y_test, y_pred, zero_division=0)

f1 = f1_score(y_test, y_pred, zero_division=0)

roc_auc = roc_auc_score(y_test, y_probability)

print("Final Model Performance")
print("=" * 35)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

Final Model Performance
Accuracy : 0.935
Precision: 0.7619
Recall   : 0.6667
F1 Score : 0.7111
ROC-AUC  : 0.9813


## Model Feature Schema

In [9]:
feature_columns = X.columns.tolist()

print("Number of features:", len(feature_columns))

print("\nFeatures:")

for feature in feature_columns:
    print(feature)

Number of features: 50

Features:
Age
Gender
Experience
Salary
Performance_Score
Job_Satisfaction
Overtime
Work_Hours
Remote_Work
Training_Hours
Projects
Promotion_Last_5Yrs
Manager_Rating
Sick_Leaves
Years_At_Company
Previous_Experience
Salary_Per_Experience
Workload_Score
Department_HR
Department_IT
Department_Marketing
Department_Sales
Job_Role_Data Analyst
Job_Role_DevOps Engineer
Job_Role_Financial Analyst
Job_Role_HR Executive
Job_Role_Marketing Specialist
Job_Role_Recruiter
Job_Role_SEO Executive
Job_Role_Sales Executive
Job_Role_Sales Manager
Job_Role_Software Engineer
Education_Master
Education_PhD
City_Islamabad
City_Karachi
City_Lahore
City_Peshawar
City_Rawalpindi
Employment_Type_Full-Time
Salary_Band_Medium
Salary_Band_High
Salary_Band_Very High
Experience_Band_Junior
Experience_Band_Mid Level
Experience_Band_Senior
Satisfaction_Category_Medium
Satisfaction_Category_High
Performance_Category_Average
Performance_Category_High


## Save Feature Schema

In [10]:
joblib.dump(feature_columns, "../models/employee_attrition_features.pkl")

print("Feature schema saved successfully.")

Feature schema saved successfully.


## Save Trained Model

In [11]:
joblib.dump(final_model, "../models/employee_attrition_model.pkl")

print("Model saved successfully.")

Model saved successfully.


## Verify Saved Model Files

In [12]:
import os

model_path = "../models/employee_attrition_model.pkl"
features_path = "../models/employee_attrition_features.pkl"

print("Model exists:", os.path.exists(model_path))

print("Feature schema exists:", os.path.exists(features_path))

Model exists: True
Feature schema exists: True


## Load Saved Model

In [13]:
loaded_model = joblib.load("../models/employee_attrition_model.pkl")

loaded_features = joblib.load("../models/employee_attrition_features.pkl")

print("Model loaded successfully.")
print("Features loaded successfully.")

Model loaded successfully.
Features loaded successfully.


## Verify Loaded Model

In [14]:
print("Number of expected features:", len(loaded_features))

print("Model feature count:", loaded_model.n_features_in_)

Number of expected features: 50
Model feature count: 50


## Test Loaded Model

In [15]:
loaded_predictions = loaded_model.predict(X_test)

loaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]

print("First 10 predictions:")

print(loaded_predictions[:10])

print("\nFirst 10 probabilities:")

print(loaded_probabilities[:10])

First 10 predictions:
[0 0 0 0 0 0 1 0 0 0]

First 10 probabilities:
[0.01188455 0.01229481 0.03969945 0.01144775 0.00168371 0.00213089
 0.63573857 0.00832877 0.05176545 0.00227208]


## Verify Model Consistency

In [16]:
predictions_match = np.array_equal(y_pred, loaded_predictions)

print("Original and loaded predictions match:", predictions_match)

Original and loaded predictions match: True


## Create Employee Attrition Prediction Function

In [17]:
def predict_employee_attrition(employee_data):

    employee_df = pd.DataFrame([employee_data])

    employee_df = employee_df[loaded_features]

    prediction = loaded_model.predict(employee_df)[0]

    probability = loaded_model.predict_proba(employee_df)[0, 1]

    if probability < 0.30:
        risk = "Low"

    elif probability < 0.60:
        risk = "Medium"

    else:
        risk = "High"

    return {"prediction": int(prediction), "probability": probability, "risk": risk}

## Create New Employee Data

In [18]:
new_employee = {}

for feature in loaded_features:
    new_employee[feature] = X_test.iloc[0][feature]

new_employee

{'Age': np.float64(60.0),
 'Gender': np.float64(1.0),
 'Experience': np.float64(15.0),
 'Salary': np.float64(216517.0),
 'Performance_Score': np.float64(4.0),
 'Job_Satisfaction': np.float64(9.0),
 'Overtime': np.float64(0.0),
 'Work_Hours': np.float64(52.0),
 'Remote_Work': np.float64(0.0),
 'Training_Hours': np.float64(45.0),
 'Projects': np.float64(11.0),
 'Promotion_Last_5Yrs': np.float64(0.0),
 'Manager_Rating': np.float64(4.0),
 'Sick_Leaves': np.float64(13.0),
 'Years_At_Company': np.float64(10.3),
 'Previous_Experience': np.float64(4.7),
 'Salary_Per_Experience': np.float64(14434.47),
 'Workload_Score': np.float64(16.2),
 'Department_HR': np.float64(0.0),
 'Department_IT': np.float64(0.0),
 'Department_Marketing': np.float64(0.0),
 'Department_Sales': np.float64(1.0),
 'Job_Role_Data Analyst': np.float64(0.0),
 'Job_Role_DevOps Engineer': np.float64(0.0),
 'Job_Role_Financial Analyst': np.float64(0.0),
 'Job_Role_HR Executive': np.float64(0.0),
 'Job_Role_Marketing Specialist':

## Predict Employee Attrition

In [19]:
result = predict_employee_attrition(new_employee)

print("Prediction Result")
print("=" * 35)

print("Attrition Prediction:", result["prediction"])

print("Attrition Probability:", round(result["probability"], 4))

print("Risk Category:", result["risk"])

Prediction Result
Attrition Prediction: 0
Attrition Probability: 0.0119
Risk Category: Low


## Human-Readable Prediction

In [21]:
if result["prediction"] == 1:
    prediction_text = "Employee is predicted to leave"
else:
    prediction_text = "Employee is predicted to stay"

print(prediction_text)

print(f"Estimated attrition probability: {result['probability']:.2%}")

print(f"Risk category: {result['risk']}")

Employee is predicted to stay
Estimated attrition probability: 1.19%
Risk category: Low


## Generate Predictions for Multiple Employees

In [22]:
sample_employees = X_test.head(10).copy()

sample_predictions = loaded_model.predict(sample_employees)

sample_probabilities = loaded_model.predict_proba(sample_employees)[:, 1]

prediction_results = sample_employees.copy()

prediction_results["Predicted_Attrition"] = sample_predictions

prediction_results["Attrition_Probability"] = sample_probabilities

In [23]:
def assign_risk(probability):

    if probability < 0.30:
        return "Low"

    elif probability < 0.60:
        return "Medium"

    else:
        return "High"


prediction_results["Risk_Category"] = prediction_results["Attrition_Probability"].apply(
    assign_risk
)

prediction_results[["Predicted_Attrition", "Attrition_Probability", "Risk_Category"]]

,Predicted_Attrition,Attrition_Probability,Risk_Category
249,0,0.011885,Low
632,0,0.012295,Low
572,0,0.039699,Low
235,0,0.011448,Low
276,0,0.001684,Low
467,0,0.002131,Low
354,1,0.635739,High
30,0,0.008329,Low
821,0,0.051765,Low
570,0,0.002272,Low


## Highest-Risk Employees

In [24]:
highest_risk = prediction_results.sort_values(
    by="Attrition_Probability", ascending=False
)

highest_risk[["Predicted_Attrition", "Attrition_Probability", "Risk_Category"]]

,Predicted_Attrition,Attrition_Probability,Risk_Category
354,1,0.635739,High
821,0,0.051765,Low
572,0,0.039699,Low
632,0,0.012295,Low
249,0,0.011885,Low
235,0,0.011448,Low
30,0,0.008329,Low
570,0,0.002272,Low
467,0,0.002131,Low
276,0,0.001684,Low


## Save Prediction Results

In [25]:
prediction_results.to_csv(
    "../data/processed/sample_prediction_results.csv", index=False
)

print("Prediction results saved successfully.")

Prediction results saved successfully.


## Create Model Metadata

In [26]:
model_metadata = {
    "model_type": "Random Forest Classifier",
    "n_estimators": 200,
    "max_depth": 8,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
    "number_of_features": len(loaded_features),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "roc_auc": roc_auc,
}

model_metadata

{'model_type': 'Random Forest Classifier',
 'n_estimators': 200,
 'max_depth': 8,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'random_state': 42,
 'number_of_features': 50,
 'accuracy': 0.935,
 'precision': 0.7619047619047619,
 'recall': 0.6666666666666666,
 'f1_score': 0.7111111111111111,
 'roc_auc': 0.9812973484848485}

In [27]:
import json

with open("../models/employee_attrition_model_metadata.json", "w") as file:
    json.dump(model_metadata, file, indent=4)

print("Model metadata saved successfully.")

Model metadata saved successfully.


## Final Model Artifacts

The trained model and supporting files are stored in the models directory.

In [28]:
import os

model_directory = "../models"

for file_name in sorted(os.listdir(model_directory)):
    print(file_name)

employee_attrition_features.pkl
employee_attrition_model.pkl
employee_attrition_model_metadata.json
